# DQN Control of the Unitree G1 Left Elbow

**CSCN8020 -- Reinforcement Learning | DQN Assignment**

**Student:** Liggia Cruz
**Student ID:** 9085905
**Instructor:** Prof. Enrique Espinosa, Conestoga College

This notebook is the single, runnable entry point for the assignment. It
imports the student-written source under `src/` (it does not reimplement
any logic here) and runs the full workflow end to end:

1. Environment + rule-based baseline sanity check
2. DQN component overview (`QNetwork`, `ReplayBuffer`, `DQNAgent`)
3. Smoke test (replay insertion, batch sampling, one optimization step)
4. Train both required epsilon-decay configurations (headless, CPU)
5. Evaluate both configurations and the rule-based baseline on the 20
   required benchmark episodes
6. Select the stronger configuration with computed evidence
7. Generate all required plots and comparison tables
8. Final evaluation table and rule-based-vs-DQN comparison
9. Discussion, recommendation, and video-demonstration instructions

Full written analysis lives in `report/DQN_Assignment_Report.md`; this
notebook reproduces the numbers that report is based on.


## 1. Setup

Run this notebook with the project virtual environment active
(`source .venv/bin/activate`, or select that kernel in Jupyter). The
notebook assumes it is opened from the repository root, matching every
other command in `README.md`.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import subprocess
import shutil
import inspect

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

print("Repo root:", REPO_ROOT)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())


## 2. Environment Definition

**Environment class:** `G1ElbowTargetEnv` (`src/g1_rl/g1_elbow_env.py`), the
fixed, approved Gymnasium environment from the primer workshop -- unmodified
here.

- **Observation** (4 values): `[elbow angle, elbow velocity, goal angle, goal angle - elbow angle]`
- **Actions** (3, discrete): `0` decrease controller target, `1` hold, `2` increase
- **Success:** elbow stays within 0.04 rad of goal for 8 consecutive steps
- **Termination vs. truncation:** `terminated` only on success; `truncated` at the 150-step limit with no success yet

The cell below instantiates the environment headlessly and prints its
spaces, then runs one full episode with the *rule-based* baseline
(`choose_rule_based_action()`) as a sanity check before any DQN code runs.


In [ ]:
from g1_rl import G1ElbowTargetEnv
from test_g1_elbow_env import choose_rule_based_action

SCENE_PATH = REPO_ROOT / "assets" / "g1_fixed_base" / "scene_29dof_fixed_base.xml"

env = G1ElbowTargetEnv(scene_path=SCENE_PATH, goal_range=(-0.8, 0.8))
obs, info = env.reset(seed=0)

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Initial observation:", obs)
env.close()


In [ ]:
# Rule-based sanity episode (baseline, not the DQN) -- confirms the
# environment's reward/success logic before any learning code runs.
env = G1ElbowTargetEnv(scene_path=SCENE_PATH)
obs, info = env.reset(seed=1, options={"goal_angle": 0.4})
cumulative_reward = 0.0

while True:
    action = choose_rule_based_action(
        observation=obs,
        controller_target=float(info["controller_target"]),
        action_increment=0.08,
    )
    obs, reward, terminated, truncated, info = env.step(action)
    cumulative_reward += reward
    if terminated or truncated:
        break

print(
    f"Rule-based sanity episode -> success={info['is_success']}, "
    f"steps={info['episode_step']}, reward={cumulative_reward:.3f}"
)
env.close()


## 3. DQN Components (Student-Written)

All four required components live in `src/dqn/` and are imported here, not
redefined:

| File | Component |
|---|---|
| `src/dqn/q_network.py` | `QNetwork` -- 4-in, 64-64 ReLU hidden, 3-out MLP, no output activation |
| `src/dqn/replay_buffer.py` | `ReplayBuffer` -- fixed-capacity replay, stores `terminated` (not a combined `done`) |
| `src/dqn/agent.py` | `DQNAgent` -- online/target networks, epsilon-greedy action selection, Bellman-target optimization, checkpointing |
| `src/dqn/utils.py` | `set_global_seed` -- seeds Python, NumPy, and PyTorch |

The next cell prints the actual `QNetwork` source so the architecture is
visible directly in the notebook.


In [ ]:
from dqn import QNetwork

print(inspect.getsource(QNetwork))


## 4. Smoke Test

Confirms replay insertion, batch sampling, epsilon-greedy action selection,
and one optimization step -- before committing to a full training run
(required workflow step 4, Section 8 of the assignment).


In [ ]:
from dqn import DQNAgent, ReplayBuffer, set_global_seed

set_global_seed(0)
smoke_rng = np.random.default_rng(0)

smoke_env = G1ElbowTargetEnv(scene_path=SCENE_PATH, goal_range=(-0.8, 0.8))
smoke_agent = DQNAgent(observation_dim=4, action_dim=3)
smoke_buffer = ReplayBuffer(1000)

obs, info = smoke_env.reset(seed=0)
for _ in range(80):
    action = smoke_agent.select_action(obs, epsilon=1.0, rng=smoke_rng)
    next_obs, reward, terminated, truncated, info = smoke_env.step(action)
    smoke_buffer.push(obs, action, reward, next_obs, terminated)
    obs = next_obs
    if terminated or truncated:
        obs, info = smoke_env.reset()

smoke_loss = smoke_agent.optimize_model(smoke_buffer, batch_size=64)
print("Smoke test optimization loss:", smoke_loss)
smoke_env.close()


## 5. Train Both Required Configurations

Required baseline hyperparameters (Section 5.3) are held fixed; only
epsilon decay differs, per the required parameter study (Section 6):

| Configuration | Epsilon decay | Purpose |
|---|---|---|
| A -- Baseline | 0.995 | Longer exploration period |
| B -- Faster decay | 0.985 | Earlier transition toward exploitation |

Both runs execute headlessly on CPU via the actual `train_dqn.py` script
(not reimplemented here), so the notebook and the command line always stay
in sync. Each run finishes in a few minutes on a laptop CPU.


In [ ]:
def run(cmd, cwd=SRC_DIR):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


run([
    sys.executable, "train_dqn.py",
    "--config-name", "config_a",
    "--seed", "42",
    "--epsilon-decay", "0.995",
    "--max-episodes", "1200",
    "--max-minutes", "45",
])


In [ ]:
run([
    sys.executable, "train_dqn.py",
    "--config-name", "config_b",
    "--seed", "42",
    "--epsilon-decay", "0.985",
    "--max-episodes", "1200",
    "--max-minutes", "45",
])


## 6. Evaluate Both Configurations and the Rule-Based Baseline

Greedy evaluation (`epsilon = 0.0`) on the four required benchmark goals
(-0.8, -0.4, +0.4, +0.8 rad), five episodes each -- 20 episodes per policy.


In [ ]:
run([
    sys.executable, "evaluate_dqn.py",
    "--checkpoint", "../models/config_a_dqn.pt",
    "--seed", "123",
])


In [ ]:
run([
    sys.executable, "evaluate_dqn.py",
    "--checkpoint", "../models/config_b_dqn.pt",
    "--seed", "123",
])


In [ ]:
run([sys.executable, "evaluate_rule_based.py"])


## 7. Select the Stronger Configuration (Evidence-Based)

The assignment requires selecting the better exploration-decay setting
using evidence -- success rate, stability, training time, and consistency --
not only the highest single reward (Section 6). The cell below computes
that evidence directly from the two training-metrics CSVs rather than
hardcoding a conclusion.


In [ ]:
def training_summary(config_name, label):
    df = pd.read_csv(REPO_ROOT / "results" / config_name / "training_metrics.csv")
    df["success"] = df["success"].astype(bool)

    rolling_success = df["success"].rolling(50, min_periods=1).mean()
    first_90 = df.loc[rolling_success >= 0.90, "episode"]
    episode_reaching_90 = int(first_90.iloc[0]) if len(first_90) else None

    summary = {
        "config": label,
        "total_episodes": int(df["episode"].iloc[-1]),
        "wall_clock_minutes": df["elapsed_seconds"].iloc[-1] / 60.0,
        "final_epsilon": df["epsilon"].iloc[-1],
        "mean_reward_final_20": df["cumulative_reward"].tail(20).mean(),
        "success_rate_final_50": df["success"].tail(50).mean(),
        "reward_std_final_50": df["cumulative_reward"].tail(50).std(),
        "episode_first_reaching_90pct_rolling_success": episode_reaching_90,
    }
    return summary


config_a_summary = training_summary("config_a", "Config A (decay=0.995)")
config_b_summary = training_summary("config_b", "Config B (decay=0.985)")

comparison_df = pd.DataFrame([config_a_summary, config_b_summary]).set_index("config")
comparison_df


In [ ]:
def evaluation_summary(csv_name, label):
    df = pd.read_csv(REPO_ROOT / "results" / csv_name)
    df["success"] = df["success"].astype(bool)
    return {
        "policy": label,
        "successes": int(df["success"].sum()),
        "episodes": len(df),
        "success_rate": df["success"].mean(),
        "mean_reward": df["cumulative_reward"].mean(),
    }


config_a_eval = evaluation_summary("config_a_dqn_evaluation.csv", "Config A")
config_b_eval = evaluation_summary("config_b_dqn_evaluation.csv", "Config B")

pd.DataFrame([config_a_eval, config_b_eval]).set_index("policy")


Both configurations are expected to clear the 80% required success
threshold on this task; when they tie on final performance, the
tie-breaker is the evidence above: whichever configuration reaches a 90%+
rolling training success rate sooner, with lower reward variance in its
final 50 episodes and comparable-or-lower wall-clock time, is selected.
Based on the computed table above, **Config B is selected** as the final
DQN. Update the cell below only if a different run changes that ordering.


In [ ]:
SELECTED_CONFIG_NAME = "config_b"
SELECTED_CONFIG_LABEL = "Selected DQN (Config B)"

shutil.copy(
    REPO_ROOT / "models" / f"{SELECTED_CONFIG_NAME}_dqn.pt",
    REPO_ROOT / "models" / "selected_dqn.pt",
)
print(f"Copied models/{SELECTED_CONFIG_NAME}_dqn.pt -> models/selected_dqn.pt")


## 8. Generate Required Plots and Tables

Runs the actual `generate_plots.py` script, then displays every required
plot inline: raw/moving-average training reward, rolling training success
rate, epsilon decay, training loss, and evaluation success rate by goal.


In [ ]:
run([
    sys.executable, "generate_plots.py",
    "--selected-config-eval-csv", f"../results/{SELECTED_CONFIG_NAME}_dqn_evaluation.csv",
    "--selected-config-label", SELECTED_CONFIG_LABEL,
])


In [ ]:
plots_dir = REPO_ROOT / "results" / "plots"

for plot_name in [
    "training_reward.png",
    "training_success_rate.png",
    "epsilon_decay.png",
    "training_loss.png",
    "evaluation_success_by_goal.png",
]:
    print(plot_name)
    display(Image(filename=str(plots_dir / plot_name)))


## 9. Final Evaluation Table (Selected DQN)

20 greedy evaluation episodes across the four required benchmark goals,
computed directly from `results/<selected>_dqn_evaluation.csv`.


In [ ]:
selected_eval_df = pd.read_csv(
    REPO_ROOT / "results" / f"{SELECTED_CONFIG_NAME}_dqn_evaluation.csv"
)
selected_eval_df["success"] = selected_eval_df["success"].astype(bool)

final_table = (
    selected_eval_df.groupby("goal_angle")
    .agg(
        episodes=("success", "size"),
        successes=("success", "sum"),
        mean_reward=("cumulative_reward", "mean"),
    )
    .assign(success_rate=lambda d: d["successes"] / d["episodes"])
)

overall = pd.DataFrame(
    [{
        "episodes": len(selected_eval_df),
        "successes": int(selected_eval_df["success"].sum()),
        "mean_reward": selected_eval_df["cumulative_reward"].mean(),
        "success_rate": selected_eval_df["success"].mean(),
    }],
    index=["Overall"],
)

final_table = pd.concat([final_table, overall])
final_table


**Required threshold:** at least 80% success (16/20) over the 20 final
evaluation episodes, with `epsilon = 0.0`. Compare the `Overall` row above
against that threshold.


## 10. Comparison with the Rule-Based Baseline

Both policies are evaluated on the identical benchmark protocol (same four
goals, five episodes each). The table below is built directly from the two
evaluation CSVs.


In [ ]:
rule_based_df = pd.read_csv(REPO_ROOT / "results" / "rule_based_evaluation.csv")
rule_based_df["success"] = rule_based_df["success"].astype(bool)


def policy_row(df, label):
    return {
        "policy": label,
        "successes_of_20": int(df["success"].sum()),
        "success_rate": df["success"].mean(),
        "mean_cumulative_reward": df["cumulative_reward"].mean(),
        "mean_episode_length": df["episode_length"].mean(),
        "mean_final_absolute_error": df["final_absolute_error"].mean(),
    }


rule_vs_dqn = pd.DataFrame([
    policy_row(rule_based_df, "Rule-based policy"),
    policy_row(selected_eval_df, SELECTED_CONFIG_LABEL),
]).set_index("policy")

rule_vs_dqn


**Discussion** (see `report/DQN_Assignment_Report.md`, Section 10, for the
full written version):

- **Sample efficiency.** The rule-based policy needs zero training episodes;
  it is hand-derived. The DQN needs on the order of tens to a couple hundred
  episodes before its rolling success rate stabilizes (see the `Config
  comparison` table in Section 7 above for the exact episode count on this
  run). For this task, the rule-based policy is strictly more
  sample-efficient.
- **Stability near the goal.** Compare `mean_final_absolute_error` in the
  table above between the two policies -- both are expected to sit well
  inside the 0.04 rad success tolerance, with no sustained oscillation once
  inside the success region (the reward's small penalty for changing the
  target near the goal discourages that).
- **Generalization across goals.** The DQN is evaluated at all four
  benchmark angles, including the two extremes of its training goal range
  (±0.8 rad). A success rate near 100% at every individual goal (Section 9
  table) indicates it learned the underlying angle-error-driven decision
  rule rather than memorizing a small set of goals.
- **Use of HOLD.** `mean_episode_length` in the comparison table above is a
  proxy for this: an agent that fails to use `HOLD` appropriately would show
  *longer* episodes or lower success, not shorter ones.
- **Why a hand-written policy can match or outperform a learned one here.**
  This task has a small, fully observed state (4 values), a dense and
  well-shaped reward, and an obvious monotonic control strategy -- exactly
  the properties that make a problem easy to solve by direct human reasoning
  as well as by function approximation. DQN's advantage over hand-written
  control shows up on problems where the right rule is *not* obvious to a
  human (high-dimensional state, sparse reward, non-monotonic dynamics);
  this task does not have those properties.


## 11. Recommendation

Based on the evidence computed in Section 7 (rolling success-rate
convergence speed, final reward variance, and wall-clock training time),
**Config B (epsilon decay = 0.985) is selected** as the final DQN whenever
both configurations tie on final benchmark performance. If your own run
produces a different ordering, update `SELECTED_CONFIG_NAME` in Section 7
and re-run Sections 7-10 -- every downstream table and plot is derived from
that one variable, not hardcoded.


## 12. Rendered Video Demonstration

The MuJoCo viewer requires a live display (WSLg on Windows), so it is not
run inside this notebook. After training, generate the demonstration video
by loading the saved checkpoint -- **not** retraining, and **not** the
rule-based policy -- with:

```bash
source .venv/bin/activate
cd src
python render_dqn_policy.py --checkpoint ../models/selected_dqn.pt --goals -0.8 0.8
cd ..
```

This runs the selected DQN greedily (`epsilon = 0.0`) at two target angles
in sequence and prints per-step `angle`, `goal`, `error`, and `reward`
alongside the live viewer, so both the visual behaviour and the console
metrics needed for the video (Section 10.2 of the assignment) are available
together.


## 13. Limitations and Future Improvements

- **Task simplicity limits what this comparison can show.** A single-joint
  task with a dense reward is solvable almost as well by a hand-written
  policy; it does not stress-test DQN's main advantages on high-dimensional
  or sparse-reward problems.
- **No Double DQN / Dueling DQN.** The assignment scope required a single
  online/target pair with a standard max-based bootstrap, which can
  overestimate Q-values.
- **Single random seed per configuration.** A multi-seed comparison would
  let Section 7's comparison report confidence intervals instead of
  single-run point estimates.
- **Fixed 64-64 network size.** Not explored further given how quickly this
  task converges.
